In [2]:
# COLAB only
!git clone https://github.com/juliaa1004/kogni_2026
!python3 -m pip install -r kogni_2026/requirements.txt
%cd kogni_2026/laboratories/lab_3

Cloning into 'kogni_2026'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 142 (delta 28), reused 135 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 4.58 MiB | 1.23 MiB/s, done.
Resolving deltas: 100% (28/28), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 104.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]
/content/kogni_2026/laboratories/lab_3


# Tokenizacja

W trakcie dzisiejszych zajęć poznamy głębiej problem tokenizacji, czyli segmentacji tekstów na tokeny. Tokeny to jednostki zgrubsza odpowiadające słowom, i.e. jednostki pomiędzy najniższym poziomem analizy (znakami), a wyższymi poziomami (np. paragrafami, chunkami, plikami).

Tokenizacja jest często pierwszym etapem pracy z tekstem. Jej znaczenie jest fundamentalne, ponieważ wszelkie niedociągnięcia jakie popełnimy na tym etapie będą propagowane w kolejnych elementach potoku przetwarzania tekstu. To na jakie segmenty podzielimy tekst, wyznacza więc granice możliwości naszej analizy. Nie zawsze decyzja ta będzie podlegała bezsprzecznym kryteriom - czasami charakterystyka danych będzie wymuszała decyzje, które w innej domenie byłyby szkodliwe.

In [3]:
import os
import pandas
import re
from pathlib import Path

Skorzystamy z danych pochodzących z serwisu https://wolnelektury.pl/

Pobraliśmy stamtąd zbiór klasycznych dramatów greckich, wpierw wykorzystajmy znaną nam metodę eksploracji katalogów, aby zebrać ścieżki do interesujących nas plików:

In [4]:
DATA_DIR = Path("../../data")
books_dir = DATA_DIR / "lektury" / "dramaty"
filepaths = []
for dir_path, subdirs, files in os.walk(books_dir):
    for file in files:
        if file.endswith(".txt"):
            filepath = Path(dir_path) / file
            print(filepath)
            filepaths.append(filepath)


../../data/lektury/dramaty/bachantki/ifigenia-w-aulidzie.txt
../../data/lektury/dramaty/bachantki/medea.txt
../../data/lektury/dramaty/bachantki/bachantki.txt
../../data/lektury/dramaty/bachantki/hippolytos-uwienczony.txt
../../data/lektury/dramaty/plaut/kupiec.txt
../../data/lektury/dramaty/plaut/bracia.txt
../../data/lektury/dramaty/plaut/zolnierz-samochwal.txt
../../data/lektury/dramaty/arystofanes/chmury.txt
../../data/lektury/dramaty/arystofanes/zaby.txt
../../data/lektury/dramaty/arystofanes/gromiwoja.txt
../../data/lektury/dramaty/sofokles/elektra.txt
../../data/lektury/dramaty/sofokles/antygona.txt
../../data/lektury/dramaty/sofokles/krol-edyp.txt
../../data/lektury/dramaty/ajschylos/oresteja.txt
../../data/lektury/dramaty/ajschylos/prometeusz-skowany.txt


Następnie wczytajmy do tabeli treść plików, i podzielmy zgrubnie każdy z tekstów korzystając z metody `split`:

In [5]:
def read_file(fpath): # helper function
    with open(fpath) as f:
        txt = f.read()
    return txt

df = pandas.DataFrame({"fpath": filepaths})
df["text"] = df["fpath"].apply(read_file)
df["pretokens"] = df["text"].apply(lambda x: x.split())

first_file = df.iloc[0]
print(first_file["text"][:200])
print(first_file["pretokens"][:50])

Eurypides

Ifigenia w Aulidzie
tłum. Jan Kasprowicz

ISBN 978-83-288-2226-9



OSOBY DRAMATU:
 * AGAMEMNON, król w Argos i Mycenach
 * MENELAOS, jego brat
 * ACHILLES, syn Peleja i Tetydy
 * STARZEC, 
['Eurypides', 'Ifigenia', 'w', 'Aulidzie', 'tłum.', 'Jan', 'Kasprowicz', 'ISBN', '978-83-288-2226-9', 'OSOBY', 'DRAMATU:', '*', 'AGAMEMNON,', 'król', 'w', 'Argos', 'i', 'Mycenach', '*', 'MENELAOS,', 'jego', 'brat', '*', 'ACHILLES,', 'syn', 'Peleja', 'i', 'Tetydy', '*', 'STARZEC,', 'sługa', 'w', 'domu', 'Agamemnona', '*', 'GONIEC', '*', 'KLITAIMNESTRA,', 'małżonka', 'Agamemnona', '*', 'IFIGENIA,', 'córka', 'Agamemnona', 'i', 'Klitaimnestry', '*', 'CHÓR', 'niewiast', 'chalcydejskich']


## Ćwiczenie 1:

Korzystając ze znanych już narzędzi biblioteki `pandas` przygotuj sumaryczną listę frekwencyjną segmentów (wyodrębnionych w kolumnie "pretokens") występujących w korpusie

In [8]:
# TODO
df["pretokens"].explode().value_counts()

,count
pretokens,
/,4230
w,4166
i,3920
się,3752
nie,3471
...,...
zachłanność,1
niezaspokojony,1
żołnierz-samochwał,1


Wykorzystajmy naszą wiedzę o regexach do konstrukcji prostego tokenizatora. Podamy mu trzy typy ciągów znaków które chcemy traktować jako tokeny:

In [12]:
simple_tokenizer = re.compile(r"\b\w+\b|"   #sequences of letters at word boundaries
                              r"\d+|"       #sequences of digits
                              r"[.,!?;—…*-]"#punctuation
                             )

df["simple_tokens"] = df["text"].apply(lambda x: [m[0] # [0] on Match  object extracts the entire matching string
                                                  for m in simple_tokenizer.finditer(x)])

Zobaczmy jak różnią się wyniki takiej tokenizacji, od wbudowanej w Pythona metody `split`:

In [10]:
first_file = df.iloc[0]
pretokens, simple_tokens = first_file["pretokens"], first_file["simple_tokens"]
for i, (ptok, stok) in enumerate(list(zip(pretokens, simple_tokens))[:100]):
    print(f"{i}\t{ptok}\t{stok}")

0	Eurypides	Eurypides
1	Ifigenia	Ifigenia
2	w	w
3	Aulidzie	Aulidzie
4	tłum.	tłum
5	Jan	.
6	Kasprowicz	Jan
7	ISBN	Kasprowicz
8	978-83-288-2226-9	ISBN
9	OSOBY	978
10	DRAMATU:	-
11	*	83
12	AGAMEMNON,	-
13	król	288
14	w	-
15	Argos	2226
16	i	-
17	Mycenach	9
18	*	OSOBY
19	MENELAOS,	DRAMATU
20	jego	*
21	brat	AGAMEMNON
22	*	,
23	ACHILLES,	król
24	syn	w
25	Peleja	Argos
26	i	i
27	Tetydy	Mycenach
28	*	*
29	STARZEC,	MENELAOS
30	sługa	,
31	w	jego
32	domu	brat
33	Agamemnona	*
34	*	ACHILLES
35	GONIEC	,
36	*	syn
37	KLITAIMNESTRA,	Peleja
38	małżonka	i
39	Agamemnona	Tetydy
40	*	*
41	IFIGENIA,	STARZEC
42	córka	,
43	Agamemnona	sługa
44	i	w
45	Klitaimnestry	domu
46	*	Agamemnona
47	CHÓR	*
48	niewiast	GONIEC
49	chalcydejskich	*
50	*	KLITAIMNESTRA
51	ARTEMIS,	,
52	bogini	małżonka
53	łowów	Agamemnona
54	Rzecz	*
55	dzieje	IFIGENIA
56	się	,
57	w	córka
58	pobliżu	Agamemnona
59	miasta	i
60	Aulis,	Klitaimnestry
61	nad	*
62	cieśniną	CHÓR
63	Eurypus,	niewiast
64	dzielącą	chalcydejskich
65	Beocję	*
66	od	ARTEMIS
67	wy

## Ćwiczenie 2:

W tekście znajdują się cztery URLe. Zmodyfikuj wyrażenie regularne wykorzystane do tokenizacji tak, aby potraktował te elementy jako pojedyncze, nierozbijalne tokeny

In [16]:
advanced_tokenizer = re.compile(r"https?://\w+\.{,3}(/\w+/?)*|"
                              r"\b\w+\b|"   #sequences of letters at word boundaries
                              r"\d+|"       #sequences of digits
                              r"[.,!?;—…*-]" #punctuation
                             )

df["simple_tokens"] = df["text"].apply(lambda x: [m[0] # [0] on Match  object extracts the entire matching string
                                                  for m in advanced_tokenizer.finditer(x)])

## Ćwiczenie 3:

Spróbuj zrekonstruować oryginalny tekst na podstawie tokenów, które ze zbadanych dotychczas tokenizatorów na to pozwalają?

In [ ]:
# TODO

## Tokenizacja BPE
W pewnych zastosowaniach istotne będzie skorzystanie z tokenizacji która rozkłada wyrazy "od spacji do spacji" na mniejsze jednostki, takie rodzaje tokenizatorow określamy często mianem "sub-word". W takim podejściu moglibyśmy np. rozbijać słowo "niejasny" na dwa tokeny: "nie" oraz "jasny" (czasami będziemy też chcieli zapamiętać informację o tym, że "nie" stanowi prefiks, w odróżnieniu od samodzielnego słowa - moglibyśmy to zasygnalizować np. zapisując "nie-")
Głównymi powodami dla którego chcielibyśmy to zrobić są:
1. ekonomia - redukcja liczności zbioru tokenów z którymi mamy do czynienia
1. wykorzystanie informacji zawartej w strukturze morfologicznej słów

Aby zobrazować tę drugą cechę spójrzmy na słowo:

*jupyterowość*

Nie występuje ono w żadnym słowniku, a jednak intuicyjnie jesteśmy w stanie powiedzieć na czym może polegać jego znaczenie. Wiedzę tę czerpiemy z morfologii tego słowa, tj. rozkładając je na dwie podjednostki: "jupyter" oraz "-owość". Każda z nich jest nam indywidualnie znajoma i dlatego, w oparciu o kompozycyjność, jesteśmy w stanie zrekonstruować zamierzone znaczenie.



Skorzystamy z dwóch dodatkowych bibliotek pomocniczych:

1. `tqdm` pozwala na opakowanie dowolnego iteratora w sposób, który sprawi że iterując przez jego elementy, wyświetli się nam pasek postępu - przydatne przy wymagających obliczeniowo zadaniach.
2. `numpy` jest biblioteką służącą do pracy z macierzami (obiekt `Array`) i algebrą liniową, wykorzystamy ją dzisiaj w podstawowym zakresie.

In [17]:
import numpy
from tqdm import tqdm

a = numpy.zeros((3,3))
print(type(a))

for row in tqdm(a):
    print(row)

<class 'numpy.ndarray'>


100%|██████████| 3/3 [00:00<00:00, 2620.89it/s]

[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]


Współczesne tokenizatory sub-word często opierają się na metodzie *Byte Pair Encoding* (**BPE**). Tokenizator taki musi zostać wytrenowany na zbiorze danych (korpusie), poprzez zastosowanie prostego algorytmu:

1. Zidentyfikuj zbiór **znaków** występujących w korpusie - będzie to początkowy zbiór tokenów
1. Przetokenizuj korpus
1. Iteratywnie:
    1. Oblicz statystyki następstwa po sobie (sukcesji) tokenów
    2. Wybierz najczęściej występującą parę tokenów
    3. dodaj do zbioru tokenów połączenie w.w. pary
    4. ponownie przetokenizuj korpus

In [20]:
char_set = sorted(set(df["text"].sum()))
token_set = [char for char in char_set] # copying
tok2ind = {c:i for i,c in enumerate(token_set)} # tokens will be mapped to indices of array
tokenizer_string = r"." # matches on every individual char

In [21]:
pretoken_counts = df["pretokens"].explode().apply(lambda x: x.strip()).value_counts()

NUM_MERGES = 600
for merge_num in tqdm(range(NUM_MERGES)):
    bpe_tokenizer = re.compile(tokenizer_string) # recreate tokenizer with new merges
    num_toks = len(token_set)
    succ_count = numpy.zeros((num_toks, num_toks)) # initialize succession array succ_count[j][k] = number of successions of j by k
    for pretok, freq in pretoken_counts.items():
        segmented = bpe_tokenizer.findall(pretok) # retokenize
        for i in range(len(segmented) - 1):
            j = tok2ind[segmented[i]]
            k = tok2ind[segmented[i+1]]
            succ_count[j][k] += freq # saving statistics

    most_freq = succ_count.argmax() # select top merge and decompose the flat index into j and k
    j = most_freq // num_toks
    k = most_freq % num_toks
    j_tok = token_set[j]
    k_tok = token_set[k]
    token_set.append(j_tok + k_tok) # add the newly merged token
    if succ_count[j][k] == succ_count[j].sum():# j_tok is ALWAYS followed by k_tok
        token_set.remove(j_tok)
    if succ_count[j][k] == succ_count.T[k].sum():# k_tok is ALWAYS preceded by j_tok
        token_set.remove(k_tok)
    tok2ind = {c:i for i,c in enumerate(token_set)}
    tokenizer_string = "|".join([re.escape(tok) for tok in sorted(token_set, key=lambda x: len(x), reverse=True)])
    # handling greediness - because of cases like "ie", "nie" and "ni" introduced in order -> "ni" will have preference over "nie"

100%|██████████| 600/600 [04:22<00:00,  2.29it/s]


Zobaczmy jak sprawuje się wytrenowany tokenizator:

In [22]:
print(bpe_tokenizer.findall("jupyterowość"))
print(bpe_tokenizer.findall("niespodziewanie"))
df["bpe_tokens"] = df["text"].apply(bpe_tokenizer.findall)

['ju', 'py', 'te', 'ro', 'wo', 'ść']
['nie', 'spo', 'dzie', 'wa', 'nie']


I porównajmy go z poprzednimi:

In [23]:
first_file = df.iloc[0]
pretokens, simple_tokens, bpe_tokens = first_file["pretokens"], first_file["simple_tokens"], first_file["bpe_tokens"]
for i, (ptok, stok, btok) in enumerate(list(zip(pretokens, simple_tokens, bpe_tokens))[:100]):
    print(f"{i}\t{ptok}\t{stok}\t{btok}")

0	Eurypides	Eurypides	E
1	Ifigenia	Ifigenia	u
2	w	w	ry
3	Aulidzie	Aulidzie	pi
4	tłum.	tłum	de
5	Jan	.	s
6	Kasprowicz	Jan	

7	ISBN	Kasprowicz	

8	978-83-288-2226-9	ISBN	I
9	OSOBY	978	fi
10	DRAMATU:	-	g
11	*	83	en
12	AGAMEMNON,	-	ia
13	król	288	 
14	w	-	w
15	Argos	2226	 
16	i	-	A
17	Mycenach	9	u
18	*	OSOBY	li
19	MENELAOS,	DRAMATU	dzie
20	jego	*	

21	brat	AGAMEMNON	t
22	*	,	łu
23	ACHILLES,	król	m.
24	syn	w	 
25	Peleja	Argos	Ja
26	i	i	n
27	Tetydy	Mycenach	 
28	*	*	Ka
29	STARZEC,	MENELAOS	sp
30	sługa	,	ro
31	w	jego	wi
32	domu	brat	cz
33	Agamemnona	*	

34	*	ACHILLES	

35	GONIEC	,	I
36	*	syn	S
37	KLITAIMNESTRA,	Peleja	B
38	małżonka	i	N
39	Agamemnona	Tetydy	 
40	*	*	9
41	IFIGENIA,	STARZEC	7
42	córka	,	8
43	Agamemnona	sługa	-
44	i	w	8
45	Klitaimnestry	domu	3
46	*	Agamemnona	-
47	CHÓR	*	2
48	niewiast	GONIEC	8
49	chalcydejskich	*	8
50	*	KLITAIMNESTRA	-
51	ARTEMIS,	,	2
52	bogini	małżonka	2
53	łowów	Agamemnona	2
54	Rzecz	*	6
55	dzieje	IFIGENIA	-
56	się	,	9
57	w	córka	

58	pobliżu	Agamemnona	

59	mi

Spójrzmy na to jakie faktycznie tokeny zostały wyuczone:

In [24]:
first_merge = len(char_set)
for i, t in enumerate(token_set[first_merge:first_merge+50]):
    print(i+1, t)

print("...")
first_merge = len(char_set)
for i, t in enumerate(token_set[len(token_set)-50:]):
    print(len(token_set) - first_merge - 49 + i, t)

1 ie
2 nie
3 sz
4 rz
5 ch
6 cz
7 na
8 st
9 dz
10 ię
11 ra
12 po
13 ro
14 ni
15 je
16 rze
17 ak
18 wa
19 ci
20 go
21 wi
22 do
23 te
24 ła
25 mi
26 li
27 wy
28 le
29 ło
30 to
31 ko
32 się
33 za
34 ny
35 wie
36 ka
37 ne
38 ty
39 rzy
40 by
41 da
42 wo
43 cie
44 że
45 dzi
46 dzie
47 ma
48 ki
49 em
50 ta
...
536 *,
537 ta,
538 nią
539 resz
540 DE
541 czą
542 słucha
543 ól
544 god
545 chwi
546 ada
547 bóg
548 kry
549 Gdzie
550 TY
551 zwy
552 ność
553 sa,
554 O,
555 EDYP
556 ANT
557 Ro
558 wy,
559 żeli
560 kim
561 tru
562 wra
563 mor
564 lub
565 ojca
566 Ka
567 gów
568 tr
569 czło
570 wiem
571 ć?
572 zaś
573 dale
574 prz
575 dy,
576 trzeba
577 j!
578 stęp
579 miło
580 tą
581 wier
582 ATE
583 Dzi
584 dzy
585 go,


Kilka obserwacji:
1. Pierwsze tokeny odzwierciedlają tak naprawdę specyfikę fonologiczną języka polskiego ("sz", "rz" etc.)
2. bardzo wcześnie pojawiają się tokeny o charakterze morfologicznym ("nie", "wy", "by")
3. później obserwujemy tokeny odpowiadające pełnym słowom ("musi", "równie", "przecież"), niektore z nich odzwierciedlają specyfikę korpusu ("PERIPLEKTOMENUS", "bóg", "Któż").

Finalnie spójrzmy na dojrzały, współczesny tokenizator wytrenowany algorytmem opartym o BPE na dużym zbiorze danych tekstowych. Jest on sparowany z modelem transformerowym o nazwie `allegro/herbert-base-cased` - polskim odpowiednikiem BERTa

https://huggingface.co/allegro/herbert-base-cased

In [25]:
from transformers import AutoTokenizer

trf_tokenizer = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
text_sample = first_file["text"][:1000]
trf_tokenized = trf_tokenizer(text_sample)
print(trf_tokenized)

config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/907k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/556k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

{'input_ids': [0, 8257, 2095, 2055, 12791, 45, 2116, 2551, 2032, 1019, 37, 2827, 2124, 18886, 1899, 3650, 14290, 15502, 29193, 29, 5879, 1750, 6108, 1750, 30231, 1750, 11406, 2979, 1750, 1017, 51, 8046, 25287, 22254, 11744, 1041, 1335, 1881, 7286, 37, 7659, 49, 6309, 1077, 1947, 13946, 1019, 2921, 25306, 1009, 3838, 23016, 1881, 7659, 8012, 9377, 18571, 1947, 2376, 11301, 1881, 37, 37435, 48, 4591, 1069, 1947, 8271, 52, 2904, 2307, 1009, 2516, 2025, 2150, 1881, 20192, 14218, 1023, 1947, 2386, 2425, 1019, 3537, 26497, 2159, 17555, 1998, 1881, 6790, 32371, 1881, 20971, 8086, 45, 49, 8012, 3834, 11196, 1947, 31747, 26497, 2159, 17555, 1998, 1881, 45, 7384, 15790, 6700, 1947, 11840, 26497, 2159, 17555, 1998, 1009, 6528, 2023, 2828, 2029, 26926, 1881, 4837, 142, 1084, 48136, 2060, 24135, 2141, 2101, 13555, 1881, 3193, 6110, 5518, 1069, 1947, 4418, 14117, 2275, 2020, 16352, 6351, 2022, 1019, 8055, 3073, 37, 2827, 1010, 1947, 2474, 2122, 3849, 2190, 8257, 2095, 20529, 1947, 2773, 5328, 2782, 

Jak widzimy jego output jest nieco bardziej złożony, i od razu dostrzegamy, że tokeny są kodowane jako liczby porządkowe. Spróbujmy zgłębić jak to kodowanie faktycznie wygląda:

In [ ]:
inv_vocab = {ind: tok for tok, ind in trf_tokenizer.vocab.items()}
for i in range(200):
    print(f"{i} -> {inv_vocab[i]}")

for i in range(2000, 2200):
    print(f"{i} -> {inv_vocab[i]}")

for i in range(20000, 20200):
    print(f"{i} -> {inv_vocab[i]}")

Zobaczmy również to kodowanie na naszym przykładzie:

In [ ]:
print(trf_tokenizer.decode(trf_tokenized["input_ids"]))
for i in trf_tokenized["input_ids"]:
    print(f"{i} -> {inv_vocab[i]}")

Ta metoda tokenizacji wiąże się z co najmniej dwoma mankamentami:

Po pierwsze: Rzadko będziemy w stanie odtworzyć oryginalny tekst z jego stokenizowanej postaci

In [ ]:
print(text_sample)
print("\n=========== vs ============\n")
print(trf_tokenizer.decode(trf_tokenized, skip_special_tokens=True))

Po drugie: nie każda sekwencja tokenów będzie sparowana z tekstem, który wygeneruje ją z powrotem (co jest konsekwencją faktu, że tokenizacja nie jest procedurą odwracalną).

In [ ]:
token_ids = [2000, 3000]
print([inv_vocab[i] for i in token_ids])
detokenized = trf_tokenizer.decode(token_ids)
print(detokenized)
retokenized = trf_tokenizer(detokenized, add_special_tokens=False)["input_ids"]
print(retokenized)
assert retokenized != token_ids
print([inv_vocab[i] for i in retokenized])

## Ćwiczenie 4
Dla każdego z tokenizatorów, i każdego z tekstów policz:
1. ile tokenów wyodrębniono z tekstu korzystając z tokenizatora
1. ile unikalnych tokenów występuje w tekście.

Wyniki przedstaw jako tabelkę korzystając z klasy `DataFrame`

In [ ]:
# TODO